In [ ]:
# SPDX-FileCopyrightText: Copyright (c) 2026 Johnny Nuñez Cano
# SPDX-License-Identifier: MIT
# Relocated from the source revision recorded in ../MIGRATION_MANIFEST.json.
# See ../LICENSE and ../THIRD_PARTY_NOTICES.md.


# CPU and GPU throughput for the two-cube box task

Compare native MuJoCo CPU execution with MuJoCo Warp on one GPU. Each environment must grasp, lift, carry and release **both cubes inside the receiving box**, settle them and withdraw the gripper. The sweep is **1, 16, 32, 64, 128, 256, 512, 1024 and 2048 environments** for each robot, with **one complete warm-up and five measured repetitions**. The complete episode lasts **40 simulated seconds**. Reports retain configuration, hardware, every repetition and physical task evidence.

This is the same box task used in Article 3, with Article 2's own pinned environment. The [updated box-task results](../benchmark-results/2026-10-09-validated-box-gpu0/README.md) cover both robots at all nine sizes on a shared AMD Ryzen Threadripper PRO 9985WX 64-Cores and one NVIDIA RTX PRO 6000 Blackwell Workstation Edition (`cuda:0`): **35 of 36 configurations passed; 1 excluded**. Every accepted configuration passed one complete warm-up and five measured episodes. See the [audited table and interpretation](../BENCHMARK.md#measured-on-the-workstation) for matched CPU/GPU timings, including transfer and validation. Excluded configurations retain their evidence and supply no accepted timing or ratio.

The [earlier October 9 box study](../benchmark-results/2026-10-09-box-sweep-rtx-pro-6000/README.md) remains available with its original configuration, timings and failures. It is separate historical evidence and is not paired with this updated sweep.

The [October 8 stacking study](../benchmark-results/2026-10-08-full-sweep-rtx-pro-6000/README.md) is historical and does not establish box-task performance.

Read the [benchmark guide](../BENCHMARK.md) for timing and acceptance details. Article 2 uses MuJoCo 3.8.0, MuJoCo Warp 3.8.0.3 and Warp 1.15.0, without Newton.


## Setup

**Local or Brev:** open this notebook from the checkout and select Article 2's Python 3.12 environment. The [setup guide](../README.md#reproduce-the-verified-environment) installs the pinned dependencies. An existing Brev VM can use the same environment and commands as a local machine.

**Google Colab:** select a GPU runtime. The setup cells download the selected source revision and install the hash-locked environment into a separate Python 3.12 virtual environment. Benchmark subprocesses use that interpreter, leaving the hosted kernel's physics packages unchanged. Before publishing results, replace `REF` with the full commit printed below and choose a fresh checkout directory.

Colab's [GPU allocation and runtime limits vary](https://research.google.com/colaboratory/faq.html). This launch path is provided for validation; the notebook alone does not establish that a device or cloud runtime has passed the task. The assigned runtime supplies its own CPU baseline. Review the full sweep's memory estimate before enabling measurements; a smaller exploratory run must be reported as a subset.


In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

# Exact imported source: physics and validation match the delivered source history.
REPO_URL = "https://github.com/johnnynunez/accelerated-computing-hub.git"
REF = "cf96302fd7200fac5941aacd9d92b1c2c7b0f386"
COLAB_CHECKOUT = Path("/content/mjwarp-benchmark-source")
TUTORIAL_RELATIVE = Path("tutorials/sim2real-blogs/notebooks/mujoco")

if IN_COLAB:
    REPO_ROOT = COLAB_CHECKOUT
    git_env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"}
    if not REPO_ROOT.exists():
        REPO_ROOT.mkdir(parents=True)
        subprocess.run(["git", "init", str(REPO_ROOT)], check=True, env=git_env)
        subprocess.run(["git", "-C", str(REPO_ROOT), "remote", "add", "origin", REPO_URL],
                       check=True, env=git_env)
        subprocess.run(["git", "-C", str(REPO_ROOT), "sparse-checkout", "init", "--cone"],
                       check=True, env=git_env)
        subprocess.run(["git", "-C", str(REPO_ROOT), "sparse-checkout", "set", str(TUTORIAL_RELATIVE)],
                       check=True, env=git_env)
    actual_remote = subprocess.check_output(
        ["git", "-C", str(REPO_ROOT), "remote", "get-url", "origin"], text=True).strip()
    if actual_remote != REPO_URL:
        raise RuntimeError("The existing checkout uses another repository; choose a fresh COLAB_CHECKOUT.")
    subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "--depth", "1", "origin", REF],
                   check=True, env=git_env)
    requested_commit = subprocess.check_output(
        ["git", "-C", str(REPO_ROOT), "rev-parse", "FETCH_HEAD"], text=True).strip()
    existing = subprocess.run(["git", "-C", str(REPO_ROOT), "rev-parse", "--verify", "HEAD"],
                              capture_output=True, text=True)
    if existing.returncode == 0 and existing.stdout.strip() != requested_commit:
        raise RuntimeError("REF changed. Existing files were preserved; choose a fresh COLAB_CHECKOUT.")
    if existing.returncode != 0:
        subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", "--detach", requested_commit],
                       check=True, env=git_env)
else:
    here = Path.cwd().resolve()
    candidates = [here, *here.parents]
    REPO_ROOT = next((p for p in candidates if
                      (p / TUTORIAL_RELATIVE / "requirements.lock.txt").is_file()), None)
    if REPO_ROOT is None:
        raise RuntimeError("Open the notebook inside the accelerated-computing-hub checkout.")

TUTORIAL_DIR = REPO_ROOT / TUTORIAL_RELATIVE
PART2 = TUTORIAL_DIR / "part2"
SCRIPT = PART2 / "migration_benchmark.py"
if not SCRIPT.is_file():
    raise RuntimeError("This source revision does not contain migration_benchmark.py.")
revision = subprocess.run(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"],
                          capture_output=True, text=True)
print("Source:", revision.stdout.strip() if revision.returncode == 0 else
      "Image snapshot without Git metadata; retain the report's source fingerprints.")
print("Tutorial:", TUTORIAL_DIR)


In [ ]:
import hashlib

# Explicit setup for a hosted runtime. Local/Brev users select their prepared kernel.
if IN_COLAB:
    runtime_root = Path("/content/mjwarp-benchmark-runtime")
    tools_env = runtime_root / "tools"
    benchmark_env = runtime_root / "python312"
    uv = tools_env / "bin/uv"
    if not uv.is_file():
        subprocess.run([sys.executable, "-m", "venv", str(tools_env)], check=True)
        subprocess.run([str(tools_env / "bin/python"), "-m", "pip", "install", "uv==0.12.5"],
                       check=True)
    BENCHMARK_PYTHON = benchmark_env / "bin/python"
    if not BENCHMARK_PYTHON.is_file():
        subprocess.run([str(uv), "venv", "--python", "3.12", str(benchmark_env)], check=True)
    subprocess.run([str(uv), "pip", "sync", "--python", str(BENCHMARK_PYTHON),
                    "--require-hashes", str(TUTORIAL_DIR / "requirements.lock.txt")],
                   check=True)
else:
    BENCHMARK_PYTHON = Path(sys.executable)

BENCHMARK_ENV = dict(os.environ)
subprocess.run([str(BENCHMARK_PYTHON), "-c",
                "import sys; print(sys.executable); print(sys.version); "
                "assert sys.version_info[:2] == (3, 12), 'Use the pinned Python 3.12 environment.'"],
               check=True, env=BENCHMARK_ENV)
lock = TUTORIAL_DIR / "requirements.lock.txt"
print("Requirements lock SHA-256:", hashlib.sha256(lock.read_bytes()).hexdigest())


## Select the workload

`TASK = "box"` selects the default benchmark. Every environment executes **2,000 control frames at 50 Hz**, each containing **20 physics steps at 0.001 s**: **40,000 steps and 40 simulated seconds**. Keep the entire task, settling period and withdrawal intact at every batch size. A smaller environment list is an exploratory subset, not a full sweep.

The CPU baseline uses persistent worker processes, capped by environment count; `CPU_THREADS` requests up to 32 workers. The same budget caps host-validation workers for both CPU and GPU results. Validation runs after simulation with the same physical checks; its history-copy, dispatch and collection costs are measured separately from simulation. Review the allocation on shared machines. CUDA advances the batch on one selected GPU. Both consume a precomputed float32 command tape, so online inverse kinematics is outside the timer.

Reset, commands, physics stepping, per-frame forward evaluation and recording of current geometry and solved contact forces belong to the simulation timer, including CPU dispatch/completion or CUDA synchronization. Setup, compilation, warm-up, final host collection/download and validation are separate costs. Both backends record **61 float32 fields per environment at 50 Hz**: time, opening, tool clearance and, for each cube, eight corners, maximum corner speed, two jaw-contact counts and two solved jaw normal forces.

Every cube must show actual loaded bilateral contact, whole-cube pickup, uninterrupted airborne carry over the box, physical opening and detachment, whole-corner containment and settling. The red-to-blue sequence and final gripper withdrawal are checked. Every warm-up and all five measured repetitions must pass; incomplete or failed configurations have no accepted timing or ratio. Both devices use 100 solver iterations, 50 line-search iterations and `impratio = 100`.

CPU and GPU use the same compiled scene and initial state. Native CPU physics is float64 and GPU physics is float32; their observation histories use the same float32 format. The [guide](../BENCHMARK.md#task-success) gives the exact task thresholds. `CPU_ONLY = True` leaves GPU performance unmeasured.

`TASK = "stack"` explicitly selects the historical 12-second stacking protocol and its different recording policy. Keep those results separate from this box sweep.

Requested capacities for the box task are `nconmax=64` and `njmax=128`. Preflight and measurement use the same flags. Reports retain actual allocated limits and contact/constraint peaks; overflow invalidates a case. Legacy stack scopes keep their own defaults. The box case timeout is 7,200 seconds; this bounds execution without shortening the simulated episode.


In [ ]:
from datetime import datetime, timezone
import shlex

TASK = "box"             # "stack" selects the historical stacking protocol
ROBOT = "so101"          # repeat with "rebot" in a separate report directory
WORLD_COUNTS = [1, 16, 32, 64, 128, 256, 512, 1024, 2048]
DEVICE = "cuda:0"        # one selected GPU per invocation
REPEATS = 5
WARMUPS = 1
CPU_THREADS = min(32, os.cpu_count() or 1)  # review your allocated CPU cores
CPU_BASELINE = "pool"    # persistent CPU processes for the box task
MAX_TRAJECTORY_MIB = 8192
CASE_TIMEOUT_SECONDS = 7200  # Complete CPU batches may take more than 30 minutes.
NCONMAX, NJMAX = 64, 128  # Requested budgets used by the validated box sweep.
CPU_ONLY = False
RUN_BENCHMARK = False    # inspect preflight and configuration, then set True

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
REPORT_DIR = PART2 / ".generated/benchmarks" / f"{TASK}-{ROBOT}-{stamp}"
common = [str(BENCHMARK_PYTHON), str(SCRIPT), "--task", TASK, "--robot", ROBOT,
          "--preset", "workstation", "--worlds", *map(str, WORLD_COUNTS),
          "--device", DEVICE, "--repeats", str(REPEATS), "--warmups", str(WARMUPS),
          "--cpu-threads", str(CPU_THREADS), "--cpu-baseline", CPU_BASELINE,
          "--max-trajectory-mib", str(MAX_TRAJECTORY_MIB)]
if TASK == "box":
    common += ["--nconmax", str(NCONMAX), "--njmax", str(NJMAX),
               "--timeout", str(CASE_TIMEOUT_SECONDS)]
if CPU_ONLY:
    common.append("--cpu-only")
command = [*common, "--output-dir", str(REPORT_DIR)]
print(shlex.join(command))


## Preflight

Inspect the selected environment, devices, configuration and observation-memory estimate before measurement. Preflight produces no performance numbers. The complete history uses **2,000 frames × 61 float32 fields per environment**: **953.125 MiB at 2,048 environments**, plus the host copy/download and additional model, solver and scratch allocations. Review CPU allocation, available RAM/VRAM and concurrent applications before enabling the full sweep. A missing GPU or capacity failure remains explicit; the notebook does not shorten the task or silently fall back to CPU.


In [ ]:
subprocess.run([*common, "--preflight", "--output-dir", str(REPORT_DIR / "preflight")],
               cwd=PART2, env=BENCHMARK_ENV, check=True)


## Measure and inspect

Setup, compilation and warm-up can take longer than a warmed run. The command prints each case and its status; detailed worker output is saved in the corresponding `.log` file in the report directory. A subprocess error must be investigated; the notebook does not substitute a successful-looking summary.

Read task success and scope before comparing throughput. Report the CPU worker count alongside speedup, and preserve individual samples. Compilation/setup cost matters when the real application only runs a short episode.


In [ ]:
if RUN_BENCHMARK:
    subprocess.run(command, cwd=PART2, env=BENCHMARK_ENV, check=True)
else:
    print("Preflight only. Set RUN_BENCHMARK = True to collect measured results.")


In [ ]:
from IPython.display import Markdown, display

summary = REPORT_DIR / "summary.md"
if (REPORT_DIR / "results.json").is_file() and summary.is_file():
    display(Markdown(summary.read_text(encoding="utf-8")))
    print("Raw reports:", REPORT_DIR)
else:
    print("No measured result from this invocation. Preflight is not a benchmark result.")


## Save the evidence

The archive below includes JSON, CSV, Markdown and worker logs from this invocation, excluding compiled caches. On Colab, set `DOWNLOAD_RESULTS = True` to download it before the runtime disconnects. Review the reports before sharing them.

For the article, retain the exact source revision, CPU/GPU/RAM, driver and package versions, world counts, CPU worker count, repetitions, timing scope, task outcomes and machine load. Ask another developer to reproduce the result on the selected accessible GPU. This notebook does not establish support or performance on untested hardware, Brev or Colab.


In [ ]:
import zipfile

DOWNLOAD_RESULTS = False
if (REPORT_DIR / "results.json").is_file():
    archive = REPORT_DIR.with_suffix(".zip")
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
        for item in sorted(REPORT_DIR.rglob("*")):
            relative = item.relative_to(REPORT_DIR)
            if (item.is_file() and item.suffix in {".json", ".csv", ".md", ".log"}
                    and ".warp-cache" not in relative.parts):
                bundle.write(item, relative)
    print("Results archive:", archive)
    if IN_COLAB and DOWNLOAD_RESULTS:
        from google.colab import files
        files.download(str(archive))
else:
    print("Run the benchmark before exporting measured results.")


## Continue

Repeat with the other robot and retain both full sweeps. Another GPU needs its own run and output directory. Keep source revision, configuration, world counts, CPU worker count and acceptance checks explicit when comparing hardware. See the [benchmark guide](../BENCHMARK.md) for interpreting CPU/GPU ratios and the [MJWarp lesson](03__mujoco_warp.ipynb) for the migration itself.
